In [ ]:
from datasets import load_dataset
dataset=load_dataset("tuetschek/atis")

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'intent', 'text', 'slots'],
        num_rows: 4978
    })
    test: Dataset({
        features: ['id', 'intent', 'text', 'slots'],
        num_rows: 893
    })
})

In [ ]:
dataset['train'][2]

{'id': 2,
 'intent': 'flight_time',
 'text': 'what is the arrival time in san francisco for the 755 am flight leaving washington',
 'slots': 'O O O B-flight_time I-flight_time O B-fromloc.city_name I-fromloc.city_name O O B-depart_time.time I-depart_time.time O O B-fromloc.city_name'}

In [ ]:
def preprocess(example):
    return{
        'tokens':example['text'].split(),
        'slot_tags':example['slots'].split(),
        'intent':example['intent']}


In [ ]:
dataset=dataset.map(preprocess)

In [ ]:
dataset['train'][0]

{'id': 0,
 'intent': 'flight',
 'text': 'i want to fly from boston at 838 am and arrive in denver at 1110 in the morning',
 'slots': 'O O O O O B-fromloc.city_name O B-depart_time.time I-depart_time.time O O O B-toloc.city_name O B-arrive_time.time O O B-arrive_time.period_of_day',
 'tokens': ['i',
  'want',
  'to',
  'fly',
  'from',
  'boston',
  'at',
  '838',
  'am',
  'and',
  'arrive',
  'in',
  'denver',
  'at',
  '1110',
  'in',
  'the',
  'morning'],
 'slot_tags': ['O',
  'O',
  'O',
  'O',
  'O',
  'B-fromloc.city_name',
  'O',
  'B-depart_time.time',
  'I-depart_time.time',
  'O',
  'O',
  'O',
  'B-toloc.city_name',
  'O',
  'B-arrive_time.time',
  'O',
  'O',
  'B-arrive_time.period_of_day']}

In [ ]:
from collections import Counter
counter=Counter()
for example in dataset['train']:
    counter.update(example['tokens'])
vocab={'PAD':0,'UNK':1}
for word in counter:
    vocab[word]=len(vocab)

In [ ]:
vocab

{'PAD': 0,
 'UNK': 1,
 'i': 2,
 'want': 3,
 'to': 4,
 'fly': 5,
 'from': 6,
 'boston': 7,
 'at': 8,
 '838': 9,
 'am': 10,
 'and': 11,
 'arrive': 12,
 'in': 13,
 'denver': 14,
 '1110': 15,
 'the': 16,
 'morning': 17,
 'what': 18,
 'flights': 19,
 'are': 20,
 'available': 21,
 'pittsburgh': 22,
 'baltimore': 23,
 'on': 24,
 'thursday': 25,
 'is': 26,
 'arrival': 27,
 'time': 28,
 'san': 29,
 'francisco': 30,
 'for': 31,
 '755': 32,
 'flight': 33,
 'leaving': 34,
 'washington': 35,
 'cheapest': 36,
 'airfare': 37,
 'tacoma': 38,
 'orlando': 39,
 'round': 40,
 'trip': 41,
 'fares': 42,
 'philadelphia': 43,
 'under': 44,
 '1000': 45,
 'dollars': 46,
 'need': 47,
 'a': 48,
 'tomorrow': 49,
 'columbus': 50,
 'minneapolis': 51,
 'kind': 52,
 'of': 53,
 'aircraft': 54,
 'used': 55,
 'cleveland': 56,
 'dallas': 57,
 'show': 58,
 'me': 59,
 'los': 60,
 'angeles': 61,
 'all': 62,
 'ground': 63,
 'transportation': 64,
 'diego': 65,
 'newark': 66,
 'by': 67,
 'way': 68,
 'houston': 69,
 "'s": 70,
 '

In [ ]:
intents=set(sample['intent'] for sample in dataset['train'])
len(intents)

22

In [ ]:
slot_set = set()
for sample in dataset['train']:
    slot_set.update(sample['slot_tags'])

In [ ]:
index2slot={index:slot for index,slot in enumerate(slot_set)}
slot2index={slot:index for index,slot in index2slot.items()}

In [ ]:
index2intent={index:intent for index,intent in enumerate(intents)}
intent2index={intent:index for index,intent in index2intent.items()}

In [ ]:
intent2index['day_name']

KeyError: 'day_name'

In [ ]:
def encode(example):
    tokens=example['tokens']
    slot_names=example['slot_tags']
    intent_labels=example['intent']
    input_ids=[vocab.get(token,1) for token in tokens]
    slot_ids=[slot2index.get(slot,0) for slot in slot_names]
    intent_id=intent2index.get(intent_labels,0)
    return {'input_ids':input_ids, 'slot_ids':slot_ids,'intent_id':intent_id}

In [ ]:
dataset=dataset.map(encode)

In [ ]:
dataset['train'][0]

{'id': 0,
 'intent': 'flight',
 'text': 'i want to fly from boston at 838 am and arrive in denver at 1110 in the morning',
 'slots': 'O O O O O B-fromloc.city_name O B-depart_time.time I-depart_time.time O O O B-toloc.city_name O B-arrive_time.time O O B-arrive_time.period_of_day',
 'tokens': ['i',
  'want',
  'to',
  'fly',
  'from',
  'boston',
  'at',
  '838',
  'am',
  'and',
  'arrive',
  'in',
  'denver',
  'at',
  '1110',
  'in',
  'the',
  'morning'],
 'slot_tags': ['O',
  'O',
  'O',
  'O',
  'O',
  'B-fromloc.city_name',
  'O',
  'B-depart_time.time',
  'I-depart_time.time',
  'O',
  'O',
  'O',
  'B-toloc.city_name',
  'O',
  'B-arrive_time.time',
  'O',
  'O',
  'B-arrive_time.period_of_day'],
 'input_ids': [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 8, 15, 13, 16, 17],
 'slot_ids': [5, 5, 5, 5, 5, 92, 5, 50, 103, 5, 5, 5, 48, 5, 23, 5, 5, 75],
 'intent_id': 10}

In [ ]:
def padding(batch):
    input_ids=[example['input_ids'] for example in batch]
    slot_ids=[example['slot_ids'] for example in batch]
    intent_ids=[example['intent_id'] for example in batch]
    maxlen=max(len(seq) for seq in input_ids)
    padded_input_ids=[]
    padded_slot_ids=[]
    masks=[]
    for input,slot in zip(input_ids,slot_ids):
        pad_len=maxlen-len(input)
        padded_input_ids.append(input+[0]*pad_len)
        padded_slot_ids.append(slot+[0]*pad_len)
        masks.append([1]*len(input)+[0]*pad_len)
    return {
    'input_ids':torch.tensor(padded_input_ids,dtype=torch.long),
    'slot_ids':torch.tensor(padded_slot_ids,dtype=torch.long),
    'intent_id':torch.tensor(intent_ids,dtype=torch.long),
    'mask':torch.tensor(masks,dtype=torch.float)}

In [ ]:
from torch.utils.data import DataLoader
train_loader=DataLoader(dataset['train'],shuffle=True,collate_fn=padding)
test_loader=DataLoader(dataset['test'],shuffle=True,collate_fn=padding)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class JointBiLSTM(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_slots, num_intents, pad_idx):
        super(JointBiLSTM, self).__init__()

        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)

        # BiLSTM
        self.bilstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        # Slot filling head (token-level)
        self.slot_fc = nn.Linear(hidden_dim * 2, num_slots)

        # Intent classification head (sentence-level)
        self.intent_fc = nn.Linear(hidden_dim * 2, num_intents)

    def forward(self, x, mask=None):
        """
        x: [batch_size, seq_len]
        mask: [batch_size, seq_len]
        """

        # 1. Embedding
        emb = self.embedding(x)   # [B, T, E]

        # 2. BiLSTM
        lstm_out, (h_n, c_n) = self.bilstm(emb)
        # lstm_out: [B, T, 2H]
        #h_n,c_n: [2,B,H]
        # 3. Slot predictions (token-wise)
        slot_logits = self.slot_fc(lstm_out)   # [B, T, num_slots]

        # 4. Intent prediction (use last hidden states)
        # h_n shape: [num_layers*2, B, H]

        forward_last = h_n[-2]   # [B, H]
        backward_last = h_n[-1]  # [B, H]

        intent_input = torch.cat((forward_last, backward_last), dim=1)  # [B, 2H]

        intent_logits = self.intent_fc(intent_input)  # [B, num_intents]

        return slot_logits, intent_logits

In [ ]:
def joint_loss(slot_logits, intent_logits, slot_labels, intent_labels, mask):
    """
    slot_logits: [B, T, num_slots]
    intent_logits: [B, num_intents]
    slot_labels: [B, T]
    intent_labels: [B]
    mask: [B, T]
    """

    # Flatten for token-level loss
    B, T, S = slot_logits.shape

    slot_logits = slot_logits.view(B * T, S)
    slot_labels = slot_labels.view(B * T)
    mask = mask.view(B * T)

    # Token loss (only valid tokens)
    slot_loss = F.cross_entropy(slot_logits, slot_labels, reduction='none')
    slot_loss = slot_loss * mask
    slot_loss = slot_loss.sum() / mask.sum()

    # Intent loss
    intent_loss = F.cross_entropy(intent_logits, intent_labels)

    return slot_loss + intent_loss

In [ ]:
def combined_accuracy(slot_logits, slot_labels, intent_logits, intent_labels, mask):

    # Slot accuracy
    slot_preds = torch.argmax(slot_logits, dim=2)   # [B, T]
    slot_correct = ((slot_preds == slot_labels).float() * mask).sum()
    slot_total = mask.sum()

    slot_acc = slot_correct / slot_total

    # Intent accuracy
    intent_preds = torch.argmax(intent_logits, dim=1)  # [B]
    intent_acc = (intent_preds == intent_labels).float().mean()

    # Combined
    return (slot_acc + intent_acc) / 2

In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = JointBiLSTM(vocab_size=len(vocab), emb_dim=128, hidden_dim=256,
                   num_slots=len(slot2index), num_intents=len(index2intent), pad_idx=0).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
from tqdm import tqdm
epochs=10
for epoch in range(epochs):
    loop=tqdm(train_loader,desc=f'Epoch:{epoch+1}/{epochs}')
    total_loss=0
    total_accuracy=0
    for batch in loop:

        input_ids = batch['input_ids'].to(device)         # [B, T]
        slot_labels = batch['slot_ids'].to(device)     # [B, T]
        intent_labels = batch['intent_id'].to(device) # [B]
        mask = batch['mask'].to(device)                  # [B, T]

        optimizer.zero_grad()

        slot_logits, intent_logits = model(input_ids, mask)

        loss = joint_loss(slot_logits, intent_logits,
                          slot_labels, intent_labels, mask)
        total_loss+=loss.item()
        total_accuracy+=combined_accuracy(slot_logits, slot_labels, intent_logits, intent_labels, mask).item()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch} Loss: {total_loss/len(train_loader)} Accuracy: {total_accuracy/len(train_loader)}")

Epoch:1/10: 100%|██████████| 4978/4978 [01:04<00:00, 77.23it/s]


Epoch 0 Loss: 0.6476745836982004 Accuracy: 0.9272288606872924


Epoch:2/10: 100%|██████████| 4978/4978 [01:00<00:00, 81.61it/s]


Epoch 1 Loss: 0.13319468175156782 Accuracy: 0.9829450212613604


Epoch:3/10: 100%|██████████| 4978/4978 [01:05<00:00, 76.33it/s]


Epoch 2 Loss: 0.052214128472586815 Accuracy: 0.993142972926674


Epoch:4/10:   9%|▉         | 463/4978 [00:06<00:59, 76.18it/s]


KeyboardInterrupt: 